# Contoh Simulasi RAW (Read After Write)



In [6]:
# Simulator from https://github.com/harshitasingh4863/instruction-level-parallelism-simulator.git

from dataclasses import dataclass

@dataclass
class Instruction:
    name: str           # Nama instruksi (contoh "I1")
    destination: str    # Register tujuan penulisan hasil (contoh "R1")
    sources: list       # Daftar register sumber yang dibaca (contoh: ["R2", "R3"])


# -------------------------------------------------
# Sample Instruction Set (contoh dengan dependensi RAW)
# -------------------------------------------------

instructions = [
    Instruction("I1", "R1", ["R2", "R3"]),     # menulis R1
    Instruction("I2", "R4", ["R1", "R5"]),     # RAW: membaca R1 dari I1
    Instruction("I3", "R6", ["R7", "R8"]),     # independen
    Instruction("I4", "R9", ["R4", "R6"]),     # RAW: membaca R4 dari I2 dan R6 dari I3
    Instruction("I5", "R10", ["R11", "R12"]),  # independen
    Instruction("I6", "R13", ["R9", "R10"]),   # RAW: membaca R9 dari I4 dan R10 dari I5
    Instruction("I7", "R14", ["R1", "R15"]),   # RAW: membaca R1 dari I1
    Instruction("I8", "R16", ["R17", "R18"]),  # independen
]


# -------------------------------------------------
# Function to check dependency
# -------------------------------------------------

def has_dependency(inst1, inst2):
    """
    Returns True if inst2 depends on inst1.
    """

    # RAW dependency:
    # inst2 membaca register yang ditulis inst1,
    # jadi inst2 harus menunggu inst1 selesai
    if inst1.destination in inst2.sources:
        return True

    return False


# -------------------------------------------------
# Sequential Execution (1 instruksi per cycle)
# -------------------------------------------------

def sequential_execution(instructions):
    print("\n" + "=" * 50)
    print("SEQUENTIAL EXECUTION")
    print("=" * 50)

    cycle = 0

    for instruction in instructions:
        cycle += 1

        print(
            f"Cycle {cycle}: Executing {instruction.name} "
            f"({instruction.destination} = "
            f"{' + '.join(instruction.sources)})"
        )

    print(f"\nTotal Sequential Cycles: {cycle}")

    return cycle


# -------------------------------------------------
# ILP / Parallel Execution
# -------------------------------------------------

def parallel_execution(instructions):
    print("\n" + "=" * 50)
    print("INSTRUCTION-LEVEL PARALLEL EXECUTION")
    print("=" * 50)

    completed = set()
    remaining = instructions.copy()

    cycle = 0

    while remaining:

        cycle += 1
        current_cycle = []

        for instruction in remaining:

            # Instruksi hanya boleh jalan jika semua instruksi
            # sebelumnya yang menulis register sumbernya (RAW)
            # sudah selesai

            dependencies_satisfied = True

            for previous in instructions:

                if previous.name == instruction.name:
                    break

                if has_dependency(previous, instruction):
                    if previous.name not in completed:
                        dependencies_satisfied = False
                        break

            if dependencies_satisfied:
                current_cycle.append(instruction)

        # Instruksi yang tidak menunggu RAW
        # dieksekusi bersamaan dalam cycle yang sama

        print(f"\nCycle {cycle}:")

        for instruction in current_cycle:
            print(
                f"  -> {instruction.name}: "
                f"{instruction.destination} = "
                f"{' + '.join(instruction.sources)}"
            )

        # Mark instructions as completed

        for instruction in current_cycle:
            completed.add(instruction.name)
            remaining.remove(instruction)

    print(f"\nTotal ILP Cycles: {cycle}")

    return cycle


# -------------------------------------------------
# Dependency Analysis
# -------------------------------------------------

def show_dependencies(instructions):

    print("\n" + "=" * 50)
    print("DEPENDENCY ANALYSIS (RAW)")
    print("=" * 50)

    found = False

    for i in range(len(instructions)):

        for j in range(i + 1, len(instructions)):

            if has_dependency(instructions[i], instructions[j]):

                print(
                    f"{instructions[j].name} depends on "
                    f"{instructions[i].name} "
                    f"(RAW, Register: {instructions[i].destination})"
                )

                found = True

    if not found:
        print("No dependencies found.")


# -------------------------------------------------
# Main Program
# -------------------------------------------------

def main():

    print("\n")
    print("*" * 60)
    print("   INSTRUCTION-LEVEL PARALLELISM SIMULATOR - RAW")
    print("*" * 60)

    print("\nInstruction Set:")

    for instruction in instructions:
        print(
            f"{instruction.name}: "
            f"{instruction.destination} = "
            f"{' + '.join(instruction.sources)}"
        )

    # Show dependencies

    show_dependencies(instructions)

    # Sequential execution

    sequential_cycles = sequential_execution(instructions)

    # Parallel execution

    parallel_cycles = parallel_execution(instructions)

    # Calculate speedup

    speedup = sequential_cycles / parallel_cycles

    efficiency = speedup / len(instructions) * 100

    print("\n" + "=" * 50)
    print("PERFORMANCE COMPARISON")
    print("=" * 50)

    print(f"Sequential Cycles : {sequential_cycles}")
    print(f"ILP Cycles        : {parallel_cycles}")
    print(f"Speedup            : {speedup:.2f}x")
    print(f"ILP Efficiency     : {efficiency:.2f}%")

    print("\nConclusion:")
    print(
        "A RAW dependency requires a later instruction "
        "to pause until an earlier instruction compltd write operation"
        "Consequently, the longest chain of dependent tasks (I1 -> I2 -> I4 -> I6) "
        "acts as the primary constraint on maximum parallel performance."
    )


# -------------------------------------------------
# Run Program
# -------------------------------------------------

if __name__ == "__main__":
    main()



************************************************************
   INSTRUCTION-LEVEL PARALLELISM SIMULATOR - RAW
************************************************************

Instruction Set:
I1: R1 = R2 + R3
I2: R4 = R1 + R5
I3: R6 = R7 + R8
I4: R9 = R4 + R6
I5: R10 = R11 + R12
I6: R13 = R9 + R10
I7: R14 = R1 + R15
I8: R16 = R17 + R18

DEPENDENCY ANALYSIS (RAW)
I2 depends on I1 (RAW, Register: R1)
I7 depends on I1 (RAW, Register: R1)
I4 depends on I2 (RAW, Register: R4)
I4 depends on I3 (RAW, Register: R6)
I6 depends on I4 (RAW, Register: R9)
I6 depends on I5 (RAW, Register: R10)

SEQUENTIAL EXECUTION
Cycle 1: Executing I1 (R1 = R2 + R3)
Cycle 2: Executing I2 (R4 = R1 + R5)
Cycle 3: Executing I3 (R6 = R7 + R8)
Cycle 4: Executing I4 (R9 = R4 + R6)
Cycle 5: Executing I5 (R10 = R11 + R12)
Cycle 6: Executing I6 (R13 = R9 + R10)
Cycle 7: Executing I7 (R14 = R1 + R15)
Cycle 8: Executing I8 (R16 = R17 + R18)

Total Sequential Cycles: 8

INSTRUCTION-LEVEL PARALLEL EXECUTION

Cycle 1:
  -> I1: 